# Machine Learning Final: Which companies would *you* apply to?
# 機械学習 期末課題：あなたが応募したい企業を予測する

In this notebook you will:

1. **Collect real job data** from Mynavi 2028 (マイナビ2028) by saving search result pages in your own browser.
2. **Extract** the company information from those pages with Beautiful Soup.
3. **Clean** the Japanese text with the Janome tokenizer.
4. **Label** each company: 1 = I'm interested, 0 = I'm not.
5. **Train** a Multinomial Naive Bayes classifier that learns your preferences.
6. **Evaluate** it with accuracy, precision, recall, and F1, and draw diagrams of the results.
7. **Write** a short report and **submit the link** to this notebook.

### Why do we save pages instead of scraping the site automatically?

Mynavi's `robots.txt` (https://job.mynavi.jp/robots.txt) tells automated programs **not** to fetch any URL that contains a `?`,
which includes every search result page, and to wait **120 seconds** between requests.
Saving the pages yourself while browsing normally respects those rules,
and your Mynavi password never goes into this notebook.

Mynavi の `robots.txt` は、`?` を含む URL（検索結果ページを含む）の自動取得を禁止しています。
そのため、この課題ではブラウザで検索結果ページを自分で保存し、そのファイルを解析します。パスワードをノートブックに入力する必要はありません。

## Step 0: Before you start
## ステップ0：はじめに

1. **Make your own copy.** Click **File → Save a copy in Drive** (ファイル → ドライブにコピーを保存). Work only in your copy; the original is read-only.
2. **How to run a cell:** click the cell, then click the ▶ button on its left or press **Shift+Enter**. Wait until the cell finishes (the ▶ stops spinning) before running the next one.
3. **Run the cells in order, from top to bottom.** Do **not** use *Runtime → Run all*: some cells wait for you to type an answer.
4. **Your work is saved in Google Drive.** Colab deletes its own files when you close the tab or stay idle for a while (about 90 minutes).
   This notebook therefore keeps your pages and labels in a Drive folder called **`ML2026_Mynavi`**.
   If Colab disconnects, just run Step 2 again, and then continue where you stopped.
5. **You need a computer** (Windows or Mac) with Chrome. Saving pages does not work on a phone.

1. **ファイル → ドライブにコピーを保存** で自分用のコピーを作成してください。
2. セルを実行するには ▶ ボタンを押すか **Shift+Enter** を押します。
3. セルは上から順番に実行してください（「すべてのセルを実行」は使わないでください）。
4. データは Google ドライブの **`ML2026_Mynavi`** フォルダに保存されます。接続が切れたら、ステップ2を再実行してから続けてください。

## Step 1: Save Mynavi search result pages (in your browser)
## ステップ1：マイナビの検索結果ページを保存する

1. Open **https://job.mynavi.jp/28/pc/** in Chrome. You may log in to your own Mynavi account, but you don't have to.
2. Search for companies. Use the search box or the menus for **企業を探す** (companies) or **インターンシップ・仕事体験** (internships), and choose conditions such as industry (業種), region (地域), or keywords.
   Pick searches that include **both** companies you like **and** companies you don't, so your model can learn the difference.
3. You are on a **search results page** when you see a list of companies and a counter like **「全5326社(1/178)」** (5,326 companies, page 1 of 178).
4. Save the page: press **Ctrl+S** (Windows) or **⌘+S** (Mac).
   - Set *Save as type* to **"Webpage, HTML Only"** (「ウェブページ、HTMLのみ」). ("Webpage, Complete" also works; you will upload only the `.html` file.)
   - **Change the file name** to `page1.html`, then `page2.html` for the next page, and so on.
     Every results page has the same default name, so if you don't rename them you will overwrite page 1 with page 2.
   - The file goes to your **Downloads** folder (ダウンロード) unless you choose another place.
5. Go to the next page: scroll down and click **「次の30社」** (next 30 companies), or click a page number. Save it as in step 4.
6. Each page has **30 companies**. Save **at least 7 pages (about 200 companies)**. More is better.

⚠️ If you were logged in, the saved files may contain your name. This notebook only reads the company cards, but **do not share the HTML files** with anyone.

1. Chrome で https://job.mynavi.jp/28/pc/ を開きます（ログインは任意です）。
2. 「企業を探す」や「インターンシップ・仕事体験」から、業種・地域・キーワードなどで検索します。興味のある企業とない企業の両方が含まれるようにしてください。
3. 「全5326社(1/178)」のような表示がある企業一覧のページが検索結果ページです。
4. **Ctrl+S**（Mac は **⌘+S**）で「ウェブページ、HTMLのみ」として保存します。**ファイル名を `page1.html`、`page2.html` … と変えてください**（同じ名前だと上書きされます）。
5. ページ下の「次の30社」またはページ番号をクリックして次のページへ進み、同じように保存します。
6. 1ページ30社です。**7ページ以上（約200社）**保存してください。

## Step 2: Set up (run this again every time Colab disconnects)
## ステップ2：準備（接続が切れたら毎回もう一度実行）

This cell:
- installs **Janome**, a Japanese tokenizer (形態素解析エンジン): https://pypi.org/project/Janome/
- installs a **Japanese font** so the diagrams can show Japanese words
- connects your **Google Drive** and creates the folder `ML2026_Mynavi`

A window will ask for permission to access your Google Drive. Choose your account and click **Allow** (許可).
It takes about one minute.

In [ ]:
!pip -q install janome
!apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/ML2026_Mynavi'
except ImportError:            # running outside Colab
    WORK_DIR = os.path.abspath('ML2026_Mynavi')

PAGES_DIR = os.path.join(WORK_DIR, 'mynavi_pages')
os.makedirs(PAGES_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Your work is saved in:', WORK_DIR)

## Step 3: Upload your saved pages
## ステップ3：保存したページをアップロード

1. Run the cell. A **Choose Files** (ファイル選択) button appears under it.
2. Click it, go to your Downloads folder, and select **all** your `page….html` files at once (hold **Ctrl**, or **⌘** on Mac, to select several). A `.zip` of the files also works.
3. Wait until every file shows **100% done**.

The files are copied to `ML2026_Mynavi/mynavi_pages` in your Drive. You can come back later, save more Mynavi pages, and run this cell again to add them.

**If the button does not work** (some browsers block it): open https://drive.google.com, go to **My Drive → ML2026_Mynavi → mynavi_pages**,
drag your `.html` files into that folder, and then run this cell and click **Cancel** (キャンセル).

In [ ]:
import shutil
import zipfile


def unique_path(folder, name):
    # Never overwrite a page that is already saved: add (2), (3), ... to the name instead
    base, ext = os.path.splitext(name)
    path, n = os.path.join(folder, name), 2
    while os.path.exists(path):
        path = os.path.join(folder, f'{base} ({n}){ext}')
        n += 1
    return path


try:
    from google.colab import files
    os.chdir('/content')                 # files.upload() saves into the current folder
    uploaded = files.upload()
    for name in uploaded:
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(name) as z:
                for member in z.namelist():
                    if member.lower().endswith(('.html', '.htm')):
                        with z.open(member) as src, open(unique_path(PAGES_DIR, os.path.basename(member)), 'wb') as dst:
                            shutil.copyfileobj(src, dst)
            os.remove(name)
        else:
            shutil.move(name, unique_path(PAGES_DIR, name))
except ImportError:
    print(f'Not running in Colab: put your saved pages in {PAGES_DIR}')
finally:
    os.chdir(WORK_DIR)

html_files = sorted(
    os.path.join(root, f)
    for root, _, names in os.walk(PAGES_DIR)
    for f in names
    if f.lower().endswith(('.html', '.htm'))
)
print(f'{len(html_files)} HTML file(s) in your mynavi_pages folder.')

## Step 4: Extract the companies with Beautiful Soup
## ステップ4：Beautiful Soup で企業情報を取り出す

Every company on a Mynavi results page is inside a `<div class="boxSearchresultEach">` box ("card").
To see it yourself, open one of your saved files in Chrome, right-click a company name, and choose **Inspect** (検証).

For each card we keep the company name, industry (業種), headquarters (本社), number of employees (従業員), and all of the card's text.
A company that appears on several pages is kept only once.

The cell prints one line per file. **Check it:** if a file says *0 new companies*, you probably saved the same page twice.

You can run this cell again after adding more pages. **Your labels from Step 6 are kept**; only new companies are added.

In [ ]:
import re
import pandas as pd
from bs4 import BeautifulSoup

CSV_FILE = '2026companies.csv'
CARD_SELECTOR = 'div.boxSearchresultEach'
CORP_LINK = re.compile(r'/search/corp(\d+)/outline\.html')
# Button and link text that appears on every card and says nothing about the company
NOISE = ['エントリー', '検討リスト登録', 'この企業の他のコース情報を見る', '説明会・面接日程']


def find_cards(soup):
    cards = soup.select(CARD_SELECTOR)
    if cards:
        return cards
    # Fallback in case Mynavi renames the class: climb up from each company link
    # until the box would contain a second company.
    cards = []
    for a in soup.find_all('a', href=CORP_LINK):
        box = a
        while box.parent is not None:
            ids = {m.group(1) for link in box.parent.find_all('a', href=CORP_LINK)
                   if (m := CORP_LINK.search(link['href']))}
            if len(ids) > 1:
                break
            box = box.parent
        if box not in cards:
            cards.append(box)
    return cards


def field(card, label):
    # Read a value such as 業種 or 本社 from the card's label list
    for li in card.select('ul.linkCtg li'):
        ttl = li.select_one('.ttl')
        if ttl and label in ttl.get_text().replace('\u3000', ''):
            return li.get_text('', strip=True).replace(ttl.get_text('', strip=True), '', 1)
    return ''


records = {}
for path in html_files:
    with open(path, encoding='utf-8', errors='ignore') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    counter = soup.select_one('p.count')
    page = re.search(r'\((\d+)/(\d+)\)', counter.get_text()) if counter else None
    cards, new = find_cards(soup), 0
    for card in cards:
        link = card.find('a', href=CORP_LINK)
        if link is None:
            continue
        corp_id = CORP_LINK.search(link['href']).group(1)
        if corp_id in records:
            continue
        text = card.get_text(' ', strip=True).replace('\u3000', '')
        for noise in NOISE:
            text = text.replace(noise, ' ')
        records[corp_id] = {
            'corp_id': corp_id,
            'company': link.get_text(strip=True),
            'industry': field(card, '業種'),
            'headquarters': field(card, '本社'),
            'employees': field(card, '従業員'),
            'Company_info': re.sub(r'\s+', ' ', text).strip(),
        }
        new += 1
    where = f'page {page.group(1)} of {page.group(2)}' if page else 'page number not found'
    warning = '   <-- no companies found. Is this a Mynavi search results page?' if not cards else \
              '   <-- 0 new companies. Did you save the same page twice?' if new == 0 else ''
    print(f'{os.path.basename(path)}: {where}, {len(cards)} companies ({new} new){warning}')

df = pd.DataFrame(records.values())
if df.empty:
    print('\nNo companies found. Did you save Mynavi *search result* pages as "Webpage, HTML Only"?')
else:
    # Keep the labels you already gave (-1 means "not labeled yet")
    df['entry?'] = -1
    if os.path.exists(CSV_FILE):
        old = pd.read_csv(CSV_FILE, encoding='utf-8', dtype={'corp_id': str})
        df['entry?'] = df['corp_id'].map(dict(zip(old['corp_id'], old['entry?']))).fillna(-1).astype(int)
    df.to_csv(CSV_FILE, index=False, encoding='utf-8')
    print(f'\n{len(df)} unique companies saved to {CSV_FILE}: '
          f'{(df["entry?"] != -1).sum()} already labeled, {(df["entry?"] == -1).sum()} still to label.')
    display(df[['company', 'industry', 'headquarters', 'employees']].head(10))

## Step 5: Clean the Japanese text
## ステップ5：日本語テキストの前処理

Janome splits Japanese text into words. We then remove:
- punctuation and symbols (品詞 = 記号, plus symbols such as / and ～)
- **stop words**: very common words such as これ, もの, ある that don't help tell companies apart.
  The list is written out in the cell below, so you can read it and change it.
- **Mynavi labels**: words such as 業種 and 応募締切 that are printed on *every* company card, so they tell us nothing about any one company.

The result is one string of space-separated words per company, which is what the classifier needs.

**Try it:** after Step 8, look at the diagram of the words most linked to each answer. If you see words that are just page decoration,
add them to `MY_STOPWORDS`, then run Steps 5, 7, and 8 again. Does the F1 score change? Mention it in your report.

In [ ]:
from janome.tokenizer import Tokenizer

# Japanese stop words from the stopwords-iso project:
# https://github.com/stopwords-iso/stopwords-ja
# Copyright (c) 2016 Gene Diaz. Released under the MIT License.
JAPANESE_STOPWORDS = '''
あそこ あっ あの あのかた あの人 あり あります ある あれ い いう います
いる う うち え お および おり おります か かつて から が
き ここ こちら こと この これ これら さ さらに し しかし する
ず せ せる そこ そして その その他 その後 それ それぞれ それで た
ただし たち ため たり だ だっ だれ つ て で でき できる
です では でも と という といった とき ところ として とともに とも と共に
どこ どの な ない なお なかっ ながら なく なっ など なに なら
なり なる なん に において における について にて によって により による に対して
に対する に関する の ので のみ は ば へ ほか ほとんど ほど ます
また または まで も もの ものの や よう より ら られ られる
れ れる を ん 何 及び 彼 彼女 我々 特に 私 私達
貴方 貴方方
'''.split()

# Labels that Mynavi prints on every company card. Add your own words here!
MY_STOPWORDS = ['業種', '本社', '従業', '員', '開催', '地域', '時期', '応募', '締切', '実施', '日数', '未満', '以上']

japanese_stopwords = set(JAPANESE_STOPWORDS) | set(MY_STOPWORDS)
print(f'Using {len(japanese_stopwords)} stop words.')

t = Tokenizer()


def clean_text(text):
    words = []
    for token in t.tokenize(str(text)):
        part_of_speech = token.part_of_speech.split(',')[0]
        if part_of_speech == '記号':           # symbol / punctuation
            continue
        if not re.search(r'\w', token.surface):  # symbols Janome missed, such as / and ～
            continue
        if token.surface in japanese_stopwords:
            continue
        words.append(token.surface)
    return ' '.join(words)


df = pd.read_csv(CSV_FILE, encoding='utf-8', dtype={'corp_id': str})
df['clean_info'] = df['Company_info'].apply(clean_text)
df.to_csv(CSV_FILE, index=False, encoding='utf-8')
print(df.loc[0, 'Company_info'][:150])
print('↓')
print(df.loc[0, 'clean_info'][:150])

## Step 6: Label the companies (1 = interested, 0 = not interested)
## ステップ6：企業にラベルを付ける（1＝応募したい、0＝応募しない）

**How to decide:** imagine you are job hunting right now. Would you click **エントリー** (apply) for this company?
Think about the industry, location, company size, and the internship or job described. Be honest and consistent; the model learns *your* taste.

**How to answer:** run the cell. A text box appears **under the cell**. Click in it, type one letter, and press **Enter**:
- **y** = yes, I would apply (1)
- **n** = no (0)
- **s** = skip this one for now
- **q** = quit and save

Every answer is saved to your Drive immediately, so you can stop at any time (type **q**, or click the ■ stop button) and run the cell again later to continue.

**Goal:** label **all** the companies, with **at least 10 "yes" and 10 "no"** answers. If almost all your answers are the same, save some more pages with different kinds of companies (Step 1) and add them (Steps 3–5).

**Alternative:** open `ML2026_Mynavi/2026companies.csv` from your Drive in Google Sheets, type 1 or 0 in the `entry?` column, download it as CSV (File → Download → .csv), and put it back in the same folder with the same name.

**判断の目安：** 今まさに就職活動をしているとして、この企業に「エントリー」しますか？業種・勤務地・規模・内容で判断してください。
セルの下に表示される入力欄に **y**（応募したい）/ **n**（応募しない）/ **s**（スキップ）/ **q**（保存して終了）を入力して Enter を押します。回答はすぐにドライブに保存されます。

In [ ]:
import textwrap

df = pd.read_csv(CSV_FILE, encoding='utf-8', dtype={'corp_id': str})
wrapper = textwrap.TextWrapper(width=60)
todo = df.index[df['entry?'] == -1]
print(f'{len(todo)} companies left to label.')

for n, idx in enumerate(todo, start=1):
    row = df.loc[idx]
    print(f'\n[{n}/{len(todo)}] {row["company"]}  |  {row["industry"]}  |  {row["headquarters"]}  |  {row["employees"]}')
    print(wrapper.fill(row['Company_info']))
    answer = input('Would you apply to this company? (y/n/s/q): ').strip().lower()
    if answer == 'q':
        break
    if answer in ('y', 'n'):
        df.loc[idx, 'entry?'] = 1 if answer == 'y' else 0
        df.to_csv(CSV_FILE, index=False, encoding='utf-8')

labeled = df[df['entry?'] != -1]
print(f'\nLabeled: {len(labeled)}  (interested: {(labeled["entry?"] == 1).sum()}, not interested: {(labeled["entry?"] == 0).sum()})')

## Step 7: Train your classifier and measure how good it is
## ステップ7：分類器を学習し、性能を評価する

**Multinomial Naive Bayes** is a light and fast classifier that works well for text.
`CountVectorizer` turns each company's words into word counts, and the classifier learns which words show up more often in the companies you chose.

We hold back **20%** of your labeled companies as a **test set** that the model never sees during training, and we measure:

| Measure | Question it answers | Formula |
|---|---|---|
| **Accuracy** | Of all test companies, how many did the model label correctly? | (TP + TN) / all |
| **Precision** | When the model says "you'd be interested", how often is it right? | TP / (TP + FP) |
| **Recall** | Of the companies you really liked, how many did the model find? | TP / (TP + FN) |
| **F1** | One number that balances precision and recall | 2 × P × R / (P + R) |

TP = true positive (you said 1, model said 1), TN = true negative (0, 0), FP = false positive (you said 0, model said 1), FN = false negative (you said 1, model said 0).
Precision, recall, and F1 are calculated for the **"interested" (1)** class.

**Baseline:** a "lazy" model that always gives your most common answer. If you said "no" to 80% of the companies, the baseline already gets 80% accuracy,
but its recall for "interested" is 0. A useful model has to beat the baseline, especially on **F1**.

**Cross-validation:** a test set of about 40 companies is small, so the scores change a lot depending on which companies land in it.
We therefore also run **5-fold cross-validation**: train and test 5 times on different parts of your data and report the **average ± standard deviation**. These are the more reliable numbers for your report.

In [ ]:
import warnings
import joblib
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

warnings.filterwarnings('ignore', category=UndefinedMetricWarning)


def new_model():
    # Japanese words can be a single character (e.g. 株, 車), so count every space-separated token
    return make_pipeline(CountVectorizer(token_pattern=r'(?u)\S+'), MultinomialNB())


def scores(y_true, y_pred):
    return [accuracy_score(y_true, y_pred),
            precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            f1_score(y_true, y_pred, pos_label=1, zero_division=0)]


df = pd.read_csv(CSV_FILE, encoding='utf-8', dtype={'corp_id': str})
if 'clean_info' not in df.columns:   # CSV edited in Sheets without the cleaned text
    df['clean_info'] = df['Company_info'].apply(clean_text)
df['clean_info'] = df['clean_info'].fillna('')
labeled = df[df['entry?'].isin([0, 1])]
counts = labeled['entry?'].value_counts()
if len(counts) < 2 or counts.min() < 5:
    raise ValueError(f'You need at least 5 companies labeled 1 AND 5 labeled 0 (you have {counts.get(1, 0)} and {counts.get(0, 0)}). Go back to Step 6.')

X, y = labeled['clean_info'], labeled['entry?']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = new_model().fit(X_train, y_train)
y_pred = model.predict(X_test)
y_base = [y_train.mode()[0]] * len(y_test)

METRICS = ['Accuracy', 'Precision', 'Recall', 'F1']
results = pd.DataFrame({'Your model': scores(y_test, y_pred), 'Baseline': scores(y_test, y_base)}, index=METRICS)

k = min(5, counts.min())
cv = cross_validate(new_model(), X, y, cv=StratifiedKFold(n_splits=k, shuffle=True, random_state=42),
                    scoring=['accuracy', 'precision', 'recall', 'f1'])
results[f'{k}-fold CV (mean ± sd)'] = [f"{cv[f'test_{m}'].mean():.2f} ± {cv[f'test_{m}'].std():.2f}"
                                       for m in ['accuracy', 'precision', 'recall', 'f1']]

print(f'Labeled companies: {len(labeled)}  (interested: {counts.get(1, 0)}, not interested: {counts.get(0, 0)})')
print(f'Training set: {len(X_train)}, test set: {len(X_test)}\n')
display(results.round(2))
print('\nDetailed report for both classes (test set):')
print(classification_report(y_test, y_pred, target_names=['not interested (0)', 'interested (1)'], zero_division=0))

# Retrain on ALL labeled companies for the final model that Step 9 uses
final_model = new_model().fit(X, y)
joblib.dump(final_model, 'company_model.pkl')

**My scores are bad. What can I do?**
- **Fewer than about 100 labeled companies:** label more. Naive Bayes needs examples.
- **Almost all answers are the same** (for example 90% "no"): the baseline is hard to beat and recall suffers. Save pages with more companies you *like* and label them.
- **Inconsistent answers:** if you said "yes" to one company and "no" to a nearly identical one, the model can't learn a rule. Look again at the companies the model got wrong.
- **Noise words:** add page-decoration words to `MY_STOPWORDS` (Step 5) and run Steps 5, 7, and 8 again.

**Error "You need at least 5 companies labeled 1 AND 5 labeled 0":** go back to Step 6 and label more companies.

It is fine if your model is not very good. What matters for your report is that you **explain** what the numbers mean and why you think the model performs the way it does.

## Step 8: Diagrams for your report
## ステップ8：レポート用の図

The cell draws three diagrams and also saves them as PNG files in your `ML2026_Mynavi` folder:

1. **Confusion matrix:** how many test companies fall into each of TP, TN, FP, FN. The diagonal (top-left to bottom-right) shows correct predictions.
2. **Your model vs. the baseline:** accuracy, precision, recall, and F1 side by side.
3. **Words most linked to each answer:** the words that push the model toward "interested" (blue) or "not interested" (orange). Do they make sense to you?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import ConfusionMatrixDisplay

# Use the Japanese font installed in Step 2
JP_FONT = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
if os.path.exists(JP_FONT):
    font_manager.fontManager.addfont(JP_FONT)
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=JP_FONT).get_name()
else:
    print('Japanese font not found: run Step 2 again if Japanese words show up as boxes.')

BLUE, ORANGE, INK, MUTED = '#2a78d6', '#eb6834', '#1f1f1e', '#6b6a63'
BLUES = LinearSegmentedColormap.from_list('blues', ['#f4f8fd', '#9ec5f4', '#2a78d6', '#0d366b'])
plt.rcParams.update({'axes.unicode_minus': False, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK, 'xtick.color': MUTED, 'ytick.color': MUTED})

# 1. Confusion matrix
fig1, ax = plt.subplots(figsize=(5.5, 4.5), layout='constrained')
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=[0, 1],
                                        display_labels=['Not interested\n(0)', 'Interested\n(1)'],
                                        cmap=BLUES, colorbar=False, ax=ax)
ax.set_xlabel('Model prediction')
ax.set_ylabel('Your answer')
ax.set_title(f'Confusion matrix (test set, {len(y_test)} companies)', color=INK, loc='left')
fig1.savefig('fig1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# 2. Model vs. baseline
fig2, ax = plt.subplots(figsize=(7, 4), layout='constrained')
x, w = np.arange(len(METRICS)), 0.38
for offset, column, color in [(-w / 2, 'Your model', BLUE), (w / 2, 'Baseline', ORANGE)]:
    bars = ax.bar(x + offset, results[column].astype(float), w - 0.04, label=column, color=color)
    ax.bar_label(bars, fmt='%.2f', padding=2, color=INK, fontsize=9)
ax.set_xticks(x, METRICS)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score (test set)')
ax.yaxis.grid(True, color='#e6e5e0')
ax.set_axisbelow(True)
ax.legend(frameon=False, loc='upper left', ncols=2)
ax.set_title('Your model vs. the always-the-same-answer baseline', color=INK, loc='left')
fig2.savefig('fig2_model_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

# 3. Words most linked to each answer (from the final model trained on all labeled companies)
vectorizer = final_model.named_steps['countvectorizer']
clf = final_model.named_steps['multinomialnb']
words = vectorizer.get_feature_names_out()
log_ratio = clf.feature_log_prob_[1] - clf.feature_log_prob_[0]   # > 0: more typical of "interested"
order = log_ratio.argsort()
top_no, top_yes = order[:10], order[::-1][:10]

fig3, ax = plt.subplots(figsize=(7, 6), layout='constrained')
idx = np.concatenate([top_no[::-1], top_yes[::-1]])
colors = [ORANGE] * len(top_no) + [BLUE] * len(top_yes)
ax.barh(range(len(idx)), log_ratio[idx], color=colors, height=0.7)
ax.set_yticks(range(len(idx)), words[idx])
ax.axvline(0, color=MUTED, linewidth=1)
ax.set_xlabel('← more typical of "not interested"     more typical of "interested" →')
ax.set_title('Words most linked to each answer', color=INK, loc='left')
fig3.savefig('fig3_top_words.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Use your model
## ステップ9：モデルを使ってみる

**9a.** Open any company page on Mynavi, copy part of its description, paste it into the box, and press **Enter**.

In [ ]:
description = input('Paste a company description: ')
probability = final_model.predict_proba([clean_text(description)])[0][1]
if probability >= 0.5:
    print(f'The model predicts you WOULD be interested ({probability:.0%}).')
else:
    print(f'The model predicts you would NOT be interested ({probability:.0%} chance of interest).')

**9b.** Rank the companies you skipped or haven't labeled yet. To get new recommendations, save more Mynavi pages and run Steps 3, 4, 5, and this cell again (you don't need to label them).

In [ ]:
df = pd.read_csv(CSV_FILE, encoding='utf-8', dtype={'corp_id': str})
if 'clean_info' not in df.columns:
    df['clean_info'] = df['Company_info'].apply(clean_text)
unlabeled = df[df['entry?'] == -1].copy()
if unlabeled.empty:
    print('Every company is labeled. Save and upload more Mynavi pages to get recommendations.')
else:
    unlabeled['interest'] = final_model.predict_proba(unlabeled['clean_info'].fillna(''))[:, 1]
    top = unlabeled.sort_values('interest', ascending=False).head(10)
    top['interest'] = top['interest'].map('{:.0%}'.format)
    top['url'] = 'https://job.mynavi.jp/28/pc/search/corp' + top['corp_id'] + '/outline.html'
    display(top[['company', 'industry', 'headquarters', 'interest', 'url']])

## Step 10: Your report (write it in the text cell below)
## ステップ10：レポート（下のテキストセルに記入）

Double-click the cell below to edit it. Answer each question in **2–4 sentences**, using the numbers and diagrams from Steps 7 and 8.
You may write in English or Japanese. (英語・日本語どちらでも構いません。)

### My report

**1. My data.** How many companies did you label, and how many were "interested" vs. "not interested"? What kinds of searches did you save?

*(your answer)*

**2. Accuracy vs. baseline.** What was your model's accuracy and the baseline's? Did your model beat the baseline? Why is accuracy alone not enough to judge this model?

*(your answer)*

**3. Precision, recall, and F1.** Give your cross-validated precision, recall, and F1 (mean ± sd). Explain in your own words what your precision and recall mean for *you* as a job hunter.

*(your answer)*

**4. Confusion matrix.** What kinds of mistakes does your model make more often: false positives or false negatives? Which kind of mistake would matter more to you, and why?

*(your answer)*

**5. Words.** Look at the words diagram. Do the words make sense for your preferences? Did you add any words to `MY_STOPWORDS`, and did that change F1?

*(your answer)*

**6. Limitations.** Name one or two reasons your model might be less accurate than it seems, or how you could improve it.

*(your answer)*

## How to submit
## 提出方法

1. Make sure you ran **every step in order** and that the scores and diagrams are visible in the notebook.
2. Check that your report (Step 10) is filled in.
3. Save: **File → Save** (ファイル → 保存), or **Ctrl+S** / **⌘+S**.
4. Click **Share** (共有) at the top right. Under *General access* choose **Anyone with the link → Viewer** (リンクを知っている全員 → 閲覧者), then click **Copy link**.
5. Submit that link.

Your saved Mynavi pages and labels stay in your own Drive; only the notebook is shared.